In [ ]:


import os
import re
import json
from pathlib import Path

import numpy as np
import pandas as pd

# ----------------------------
# 1) CONFIG
# ----------------------------
DATA_DIR = Path("")
INPUT_FILENAME = "data.xlsx" 

# Your exact column names (must match file header)
EXPECTED_COLUMNS = [
    "Rank", "University", "Country", "Program", "official title", "Degree Level",
    "Launch Year / Period", "Language of Instruction", "Duration", "Mode of Delivery",
    "Primary Field", "Subfield(s)", "Interdisciplinary Linkages", "Key Technical Skills",
    "Analytical Skills", "Soft / Professional Skills", "Research Capabilities",
    "Objectives", "Drivers", "Audiences", "Career Pathways", "Industrial Linkage Evidence",
    "Limitations", "Innovative Aspects", "Relevance for Iran"
]

# Text-heavy columns (for your later NLP pipeline)
TEXT_COLUMNS = [
    "Program", "official title", "Primary Field", "Subfield(s)", "Interdisciplinary Linkages",
    "Key Technical Skills", "Analytical Skills", "Soft / Professional Skills", "Research Capabilities",
    "Objectives", "Drivers", "Audiences", "Career Pathways", "Industrial Linkage Evidence",
    "Limitations", "Innovative Aspects", "Relevance for Iran"
]

# Categorical columns (light cleaning only)
CATEGORICAL_COLUMNS = [
    "University", "Country", "Degree Level", "Language of Instruction", "Duration", "Mode of Delivery"
]

# Numeric-like columns
NUMERIC_COLUMNS = ["Rank"]

# Cells that should be treated as missing (NaN) — without filling
MISSING_TOKENS = {
    "", " ", "  ", "\t", "\n", "\r",
    "-", "—", "–", "_",
    "none", "None", "NONE",
    "null", "NULL", "Null",
    "nan", "NaN", "NAN",
    "n/a", "N/A", "na", "NA",
}

# Regex for multiple whitespace
RE_MULTI_WS = re.compile(r"\s+")

# ----------------------------
# 2) HELPERS
# ----------------------------
def find_input_file(data_dir: Path, forced_name: str | None = None) -> Path:
    if forced_name:
        p = data_dir / forced_name
        if not p.exists():
            raise FileNotFoundError(f"INPUT_FILENAME was set but file not found: {p}")
        return p

    candidates = []
    for ext in ("*.xlsx", "*.xls", "*.csv"):
        candidates.extend(sorted(data_dir.glob(ext)))

    if not candidates:
        raise FileNotFoundError(f"No .xlsx/.xls/.csv files found in: {data_dir}")

    # pick the most recently modified file
    candidates.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    return candidates[0]


def normalize_missing(x):
    """
    Convert tokens like '-', 'none', '', etc. to np.nan.
    IMPORTANT: does not fill missing; only standardizes.
    """
    if x is None:
        return np.nan
    if isinstance(x, float) and np.isnan(x):
        return np.nan
    if isinstance(x, str):
        s = x.strip()
        if s in MISSING_TOKENS:
            return np.nan
        # also treat strings that become missing after stripping punctuation-like dashes
        if s.lower() in {t.lower() for t in MISSING_TOKENS}:
            return np.nan
        return x
    return x


def clean_text_keep_nan(x):
    """
    Clean text ONLY if not missing.
    - strip edges
    - collapse whitespace
    - keep original case (recommended for program titles) OR set lower() if you want
    """
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.nan

    # If numeric accidentally ended up in a text col, keep as string (but do not fabricate)
    s = str(x)

    s = s.strip()
    if s in MISSING_TOKENS or s.lower() in {t.lower() for t in MISSING_TOKENS}:
        return np.nan

    # normalize whitespace
    s = RE_MULTI_WS.sub(" ", s)

    # remove invisible unicode spaces
    s = s.replace("\u200c", " ").replace("\u200f", " ").replace("\ufeff", "")

    # final trim
    s = s.strip()
    if not s or s in MISSING_TOKENS or s.lower() in {t.lower() for t in MISSING_TOKENS}:
        return np.nan
    return s


def coerce_rank_to_int_keep_nan(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.nan
    if isinstance(x, (int, np.integer)):
        return int(x)
    if isinstance(x, float):
        # 1.0 -> 1
        if np.isfinite(x):
            return int(x)
        return np.nan
    s = str(x).strip()
    if s in MISSING_TOKENS or s.lower() in {t.lower() for t in MISSING_TOKENS}:
        return np.nan
    # extract digits
    m = re.search(r"\d+", s)
    if not m:
        return np.nan
    return int(m.group(0))


def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Make column names consistent:
    - strip
    - collapse multiple spaces
    - keep original wording as much as possible
    """
    df = df.copy()
    new_cols = []
    for c in df.columns:
        cc = str(c).strip()
        cc = RE_MULTI_WS.sub(" ", cc)
        new_cols.append(cc)
    df.columns = new_cols
    return df


def validate_expected_columns(df: pd.DataFrame):
    missing = [c for c in EXPECTED_COLUMNS if c not in df.columns]
    extra = [c for c in df.columns if c not in EXPECTED_COLUMNS]
    # We do not hard-fail on extra columns; but we warn.
    return missing, extra


# ----------------------------
# 3) LOAD
# ----------------------------
input_path = find_input_file(DATA_DIR, INPUT_FILENAME)

if input_path.suffix.lower() in [".xlsx", ".xls"]:
    df = pd.read_excel(input_path, dtype=object)  # dtype=object to preserve everything
elif input_path.suffix.lower() == ".csv":
    # keep_default_na=False so pandas doesn't auto-convert tokens; we handle ourselves
    df = pd.read_csv(input_path, dtype=object, keep_default_na=False, encoding="utf-8")
else:
    raise ValueError(f"Unsupported file type: {input_path.suffix}")

df = standardize_columns(df)

# ----------------------------
# 4) BASIC CHECKS
# ----------------------------
report = {
    "input_file": str(input_path),
    "rows_original": int(df.shape[0]),
    "cols_original": int(df.shape[1]),
    "warnings": [],
}

missing_cols, extra_cols = validate_expected_columns(df)
if missing_cols:
    report["warnings"].append(
        f"Missing expected columns (check header spelling): {missing_cols}"
    )
if extra_cols:
    report["warnings"].append(
        f"Extra columns present (kept as-is): {extra_cols}"
    )

# Ensure expected columns exist (create as NaN if absent, but do NOT fill with content)
# This does not violate your rule; it only preserves schema for downstream pipeline.
for c in EXPECTED_COLUMNS:
    if c not in df.columns:
        df[c] = np.nan

# Reorder columns
df = df[EXPECTED_COLUMNS + [c for c in df.columns if c not in EXPECTED_COLUMNS]]

# ----------------------------
# ----------------------------
# 5) NORMALIZE MISSING TOKENS (NO IMPUTATION)
# ----------------------------
for col in df.columns:
    df[col] = df[col].map(normalize_missing)


# ----------------------------
# 6) TYPE-SAFE CLEANING
# ----------------------------
# Rank -> numeric (keep NaN)
if "Rank" in df.columns:
    df["Rank"] = df["Rank"].apply(coerce_rank_to_int_keep_nan)

# Clean categorical columns (light text normalization, keep NaN)
for c in CATEGORICAL_COLUMNS:
    if c in df.columns:
        df[c] = df[c].apply(clean_text_keep_nan)

# Clean text columns (for NLP later), keep NaN
for c in TEXT_COLUMNS:
    if c in df.columns:
        df[c] = df[c].apply(clean_text_keep_nan)

# Optional: strip whitespace for all remaining object columns, without changing NaN
for c in df.columns:
    if df[c].dtype == "object" and c not in (TEXT_COLUMNS + CATEGORICAL_COLUMNS):
        df[c] = df[c].apply(clean_text_keep_nan)

# ----------------------------
# 7) CONSISTENCY CHECKS (NON-DESTRUCTIVE)
# ----------------------------
# Do not drop rows; only report.
na_counts = df.isna().sum().to_dict()
report["missing_counts_per_column"] = {k: int(v) for k, v in na_counts.items()}

# Rank uniqueness / range quick checks
if "Rank" in df.columns:
    finite_ranks = df["Rank"].dropna().astype(int)
    report["rank_min"] = int(finite_ranks.min()) if not finite_ranks.empty else None
    report["rank_max"] = int(finite_ranks.max()) if not finite_ranks.empty else None
    report["rank_unique_count"] = int(finite_ranks.nunique()) if not finite_ranks.empty else 0

# Row count expectation check
if df.shape[0] not in (491, 492):  # you said 492 total including header => 491 data rows, but some files include 492 data rows
    report["warnings"].append(
        f"Row count is {df.shape[0]}. You expected ~491/492. Verify file export."
    )

# ----------------------------
# 8) SAVE OUTPUTS
# ----------------------------
out_parquet = DATA_DIR / "cleaned_dataset.parquet"
out_xlsx = DATA_DIR / "cleaned_dataset.xlsx"
out_csv = DATA_DIR / "cleaned_dataset.csv"
out_report = DATA_DIR / "preprocessing_report.json"

# Parquet preserves NaN + types best
df.to_parquet(out_parquet, index=False)

# Excel/CSV for convenience (NaN becomes empty cells)
df.to_excel(out_xlsx, index=False)
df.to_csv(out_csv, index=False, encoding="utf-8-sig")

with open(out_report, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("Done.")
print(f"Input:  {input_path}")
print(f"Saved:  {out_parquet}")
print(f"Saved:  {out_xlsx}")
print(f"Saved:  {out_csv}")
print(f"Report: {out_report}")

if report["warnings"]:
    print("\nWARNINGS:")
    for w in report["warnings"]:
        print("-", w)


import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# ----------------------------
# Imports
# ----------------------------
import re
import json
from pathlib import Path
from math import factorial

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sentence_transformers import SentenceTransformer


BASE_DIR = Path("")
INPUT_PARQUET = BASE_DIR / "cleaned_dataset.parquet"

OUT_DIR = BASE_DIR / "outputs_methodology"
OUT_DIR.mkdir(parents=True, exist_ok=True)

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
BATCH_SIZE = 64
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Columns used to build the corpus (only non-missing values)
CORPUS_COLS = [
    "Drivers",
    "Objectives",
    "Innovative Aspects",
    "Key Technical Skills",
    "Analytical Skills",
    "Soft / Professional Skills",
    "Career Pathways",
    "Industrial Linkage Evidence",
    "Interdisciplinary Linkages",
]

# Semantic axes with seed phrases
AXES = {
    "AI": [
        "machine learning", "deep learning", "neural networks", "foundation models",
        "large language models", "natural language processing", "computer vision",
        "data mining", "predictive analytics", "reinforcement learning", "MLOps",
        "generative AI", "AI ethics", "explainable AI", "knowledge graphs"
    ],
    "SUS": [
        "sustainability", "climate change", "net zero", "decarbonization",
        "renewable energy", "carbon management", "circular economy",
        "environmental resilience", "ESG", "green technologies",
        "climate adaptation", "biodiversity", "clean energy transition"
    ],
    "HEALTH": [
        "digital health", "biomedical engineering", "clinical AI",
        "health informatics", "medical data", "public health",
        "telemedicine", "precision medicine", "wearables",
        "patient monitoring", "bioinformatics", "computational biology"
    ],
    "CYBER": [
        "cybersecurity", "information security", "privacy", "data governance",
        "AI governance", "regulation", "policy", "risk management",
        "trustworthy AI", "responsible AI", "ethics", "compliance",
        "law and technology"
    ],
    "INDUSTRY": [
        "industry partnership", "industrial collaboration", "internship",
        "capstone project", "industry-led", "professional practice",
        "employability", "work-integrated learning", "technology transfer",
        "startup", "entrepreneurship", "innovation ecosystem"
    ],
}

DRIVER_SPLIT_REGEX = re.compile(
    r"(?:;|\||/|,|\band\b|\bor\b|\n|\r)+",
    flags=re.IGNORECASE
)

TOPK_DRIVERS_PER_AXIS = 5
TOPK_COUNTRY_DRIVERS = 15

# ----------------------------
# 1) HELPERS
# ----------------------------
def is_missing(x) -> bool:
    return x is None or (isinstance(x, float) and np.isnan(x))


def safe_join_text(row: pd.Series, cols: list[str]) -> str | None:
    parts = []
    for c in cols:
        if c not in row.index:
            continue
        v = row[c]
        if isinstance(v, str) and v.strip():
            parts.append(f"{c}: {v.strip()}")
        elif not is_missing(v):
            parts.append(f"{c}: {str(v).strip()}")
    return " | ".join(parts) if parts else None


def cosine_sim_matrix(A: np.ndarray, b: np.ndarray) -> np.ndarray:
    A_norm = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-12)
    b_norm = b / (np.linalg.norm(b) + 1e-12)
    return A_norm @ b_norm


def minmax_scale_preserve_nan(x: np.ndarray) -> np.ndarray:
    out = np.full_like(x, np.nan, dtype=float)
    mask = np.isfinite(x)
    if mask.any():
        out[mask] = MinMaxScaler().fit_transform(x[mask].reshape(-1, 1)).ravel()
    return out


def coalition_value(scores: np.ndarray) -> float:
    if scores.size == 0:
        return 0.0
    scores = np.clip(scores.astype(float), 0.0, 1.0)
    return float(1.0 - np.prod(1.0 - scores))


def shapley_values(axis_scores: dict[str, float]) -> dict[str, float]:
    axes = list(axis_scores.keys())
    n = len(axes)

    s = np.array([
        0.0 if is_missing(v) else float(v)
        for v in axis_scores.values()
    ])

    v = {}
    for mask in range(1 << n):
        subset = [s[i] for i in range(n) if mask & (1 << i)]
        v[mask] = coalition_value(np.array(subset))

    shap = np.zeros(n)
    n_fact = factorial(n)

    for i in range(n):
        for mask in range(1 << n):
            if mask & (1 << i):
                continue
            k = bin(mask).count("1")
            weight = factorial(k) * factorial(n - k - 1) / n_fact
            shap[i] += weight * (v[mask | (1 << i)] - v[mask])

    return {axes[i]: float(shap[i]) for i in range(n)}


def parse_driver_phrases(text) -> list[str]:
    if is_missing(text):
        return []
    chunks = [c.strip() for c in DRIVER_SPLIT_REGEX.split(str(text)) if c.strip()]
    seen, out = set(), []
    for c in chunks:
        if c.lower() not in seen:
            seen.add(c.lower())
            out.append(c)
    return out

# ----------------------------
# 2) LOAD
# ----------------------------
df = pd.read_parquet(INPUT_PARQUET)

# ----------------------------
# 3) STEP 1: TEXT_CORPUS
# ----------------------------
df["TEXT_CORPUS"] = df.apply(lambda r: safe_join_text(r, CORPUS_COLS), axis=1)
mask_corpus = df["TEXT_CORPUS"].notna()
df_corpus = df.loc[mask_corpus].copy()

# ----------------------------
# 4) STEP 2: PROGRAM EMBEDDINGS
# ----------------------------
model = SentenceTransformer(EMBED_MODEL_NAME)

emb_program = model.encode(
    df_corpus["TEXT_CORPUS"].tolist(),
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True
)

# ----------------------------
# 5) STEP 3: SEEDED AXES
# ----------------------------
axis_vectors = {}
for axis, seeds in AXES.items():
    axis_vectors[axis] = model.encode(seeds, convert_to_numpy=True).mean(axis=0)

axis_scores = {}
for axis, vec in axis_vectors.items():
    sims = cosine_sim_matrix(emb_program, vec)
    axis_scores[axis] = minmax_scale_preserve_nan(sims)
    df_corpus[f"score_{axis}"] = axis_scores[axis]

df_scores = df.copy()
for axis in AXES:
    df_scores[f"score_{axis}"] = np.nan
    df_scores.loc[df_corpus.index, f"score_{axis}"] = df_corpus[f"score_{axis}"]

# ----------------------------
# 6) STEP 4: GAME THEORY
# ----------------------------
axis_names = list(AXES.keys())
emergence = []
shapley_all = {a: [] for a in axis_names}

for _, row in df_scores.iterrows():
    scores = {a: row.get(f"score_{a}", np.nan) for a in axis_names}
    s = np.array([0.0 if is_missing(v) else v for v in scores.values()])
    emergence.append(coalition_value(s))

    shap = shapley_values(scores)
    for a in axis_names:
        shapley_all[a].append(shap[a])

df_scores["EmergenceScore"] = emergence
for a in axis_names:
    df_scores[f"Shapley_{a}"] = shapley_all[a]

# ----------------------------
# 7) STEP 5: DRIVER ATTRIBUTION
# ----------------------------
driver_rows = []

for idx, row in df_scores.iterrows():
    phrases = parse_driver_phrases(row.get("Drivers"))
    if not phrases:
        continue

    emb_d = model.encode(phrases, convert_to_numpy=True)

    for axis, vec in axis_vectors.items():
        sims = cosine_sim_matrix(emb_d, vec)
        for rank, j in enumerate(np.argsort(-sims)[:TOPK_DRIVERS_PER_AXIS], 1):
            driver_rows.append({
                "row_index": idx,
                "Country": row.get("Country"),
                "University": row.get("University"),
                "Program": row.get("Program"),
                "Axis": axis,
                "DriverPhrase": phrases[j],
                "DriverSimRaw": float(sims[j]),
                "TopRank": rank,
            })

df_driver = pd.DataFrame(driver_rows)

df_country = (
    df_driver
    .groupby(["Country", "Axis", "DriverPhrase"])
    .agg(mean_sim=("DriverSimRaw", "mean"), count=("DriverSimRaw", "count"))
    .reset_index()
    .sort_values(["Country", "Axis", "mean_sim", "count"], ascending=[True, True, False, False])
)

df_country["rank"] = df_country.groupby(["Country", "Axis"]).cumcount() + 1
df_country = df_country[df_country["rank"] <= TOPK_COUNTRY_DRIVERS]

# ----------------------------
# 8) SAVE OUTPUTS
# ----------------------------
df_scores.to_parquet(OUT_DIR / "methodology_program_scores.parquet", index=False)
df_scores.to_excel(OUT_DIR / "methodology_program_scores.xlsx", index=False)
df_driver.to_excel(OUT_DIR / "driver_attribution_top_per_program.xlsx", index=False)
df_country.to_excel(OUT_DIR / "driver_attribution_country_summary.xlsx", index=False)

with open(OUT_DIR / "methodology_run_report.json", "w", encoding="utf-8") as f:
    json.dump({
        "rows_total": int(df_scores.shape[0]),
        "rows_with_text_corpus": int(mask_corpus.sum()),
        "embedding_model": EMBED_MODEL_NAME,
        "axes": axis_names
    }, f, indent=2)

print("DONE (methodology only).")


# ============================================================
# IMPORTS
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import geopandas as gpd
import pycountry

from pathlib import Path
from itertools import combinations
from matplotlib.patches import Polygon

# ============================================================
# CONFIG
# ============================================================
BASE_DIR = Path(r"")

DATA_PATH = BASE_DIR / "outputs_methodology" / "methodology_program_scores.parquet"
WORLD_SHP = BASE_DIR / "world_map" / "ne_110m_admin_0_countries.shp"

FIG_DIR = BASE_DIR / "figures_q1"
FIG_DIR.mkdir(exist_ok=True, parents=True)

DPI = 300

AXES = ["AI", "SUS", "HEALTH", "CYBER", "INDUSTRY"]
YEAR_COL = "Launch Year / Period"

AXIS_COLORS = {
    "AI": "#d62728",
    "SUS": "#2ca02c",
    "HEALTH": "#1f77b4",
    "CYBER": "#9467bd",
    "INDUSTRY": "#ff7f0e",
}

# ============================================================
# BASIC VALIDATION
# ============================================================
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Parquet not found: {DATA_PATH}")

if not WORLD_SHP.exists():
    raise FileNotFoundError(f"Shapefile not found: {WORLD_SHP}")

# ============================================================
# LOAD DATA
# ============================================================
df = pd.read_parquet(DATA_PATH)
df_eval = df.copy()

required_cols = ["Program", "Country", "EmergenceScore", YEAR_COL] + [f"score_{a}" for a in AXES]
missing = [c for c in required_cols if c not in df_eval.columns]
if missing:
    raise ValueError(f"Missing required columns in parquet: {missing}")

# ============================================================
# COUNTRY → ISO3 HELPERS
# ============================================================
ISO3_OVERRIDES = {
    "United States": "USA",
    "United States of America": "USA",
    "UK": "GBR",
    "United Kingdom": "GBR",
    "Russia": "RUS",
    "Iran": "IRN",
    "South Korea": "KOR",
    "North Korea": "PRK",
    "Vietnam": "VNM",
    "Czech Republic": "CZE",
    "Czechia": "CZE",
}

def country_to_iso3(name):
    if pd.isna(name):
        return None
    s = str(name).strip()
    if not s:
        return None
    if s in ISO3_OVERRIDES:
        return ISO3_OVERRIDES[s]
    try:
        return pycountry.countries.lookup(s).alpha_3
    except Exception:
        return None

# ============================================================
# 1) NLP EVALUATION — AXIS SEPARABILITY
# ============================================================
axis_scores = df_eval[[f"score_{a}" for a in AXES]].dropna(how="all")
axis_corr = axis_scores.corr()

plt.figure(figsize=(7, 6))
sns.heatmap(axis_corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Axis Separability (Correlation of Axis Scores)")
plt.tight_layout()
plt.savefig(FIG_DIR / "axis_separability_heatmap.png", dpi=DPI, bbox_inches="tight")
plt.close()

# ============================================================
# 2) GAME THEORY EVALUATION
# ============================================================
# (Keep if Shapley_* exists; otherwise skip safely)
shapley_cols = [f"Shapley_{a}" for a in AXES]
if all(c in df_eval.columns for c in shapley_cols):
    df_eval["Shapley_sum"] = df_eval[shapley_cols].sum(axis=1, min_count=1)
    df_eval["Shapley_residual"] = df_eval["EmergenceScore"] - df_eval["Shapley_sum"]
    df_eval["Shapley_residual"].describe().to_csv(FIG_DIR / "shapley_consistency_stats.csv")
else:
    pd.Series(
        {"note": "Shapley_* columns not found; shapley consistency step skipped."}
    ).to_csv(FIG_DIR / "shapley_consistency_stats.csv")

df_eval["MeanAxisScore"] = df_eval[[f"score_{a}" for a in AXES]].mean(axis=1)
df_eval["Synergy"] = df_eval["EmergenceScore"] - df_eval["MeanAxisScore"]

plt.figure(figsize=(7, 4))
sns.histplot(df_eval["Synergy"].dropna(), bins=30, kde=True)
plt.title("Synergy Effect Distribution (EmergenceScore − Mean Axis Score)")
plt.tight_layout()
plt.savefig(FIG_DIR / "synergy_distribution.png", dpi=DPI, bbox_inches="tight")
plt.close()

# ============================================================
# ============================================================
# 3) NLP GRAPH — READABLE Program–Axis Bipartite Network
# ============================================================

EDGE_THRESHOLD = 0.65
TOP_PROGRAMS = 120

df_graph = (
    df_eval[["Program", "EmergenceScore"] + [f"score_{a}" for a in AXES]]
    .dropna(subset=["Program"])
    .sort_values("EmergenceScore", ascending=False)
    .head(TOP_PROGRAMS)
)

# ----------------------------
# Build Graph
# ----------------------------
G = nx.Graph()

for a in AXES:
    G.add_node(a, node_type="axis")

for _, row in df_graph.iterrows():
    prog = str(row["Program"]).strip()
    if not prog:
        continue

    G.add_node(prog, node_type="program")

    for a in AXES:
        s = row.get(f"score_{a}")
        if pd.notna(s) and float(s) >= EDGE_THRESHOLD:
            G.add_edge(prog, a, weight=float(s), axis=a)

# ----------------------------
# Layout (Axis ring + program jitter)
# ----------------------------
pos = {}
theta = np.linspace(0, 2 * np.pi, len(AXES), endpoint=False)

for a, t in zip(AXES, theta):
    pos[a] = np.array([np.cos(t), np.sin(t)])

rng = np.random.default_rng(42)

for n, d in G.nodes(data=True):
    if d.get("node_type") == "program":
        edges = list(G.edges(n, data=True))
        if edges:
            best_axis = max(edges, key=lambda x: x[2]["weight"])[1]
            pos[n] = pos[best_axis] + rng.normal(scale=0.25, size=2)
        else:
            pos[n] = rng.normal(scale=0.3, size=2)

# ----------------------------
# Plot
# ----------------------------
plt.figure(figsize=(14, 14))

# Program nodes
nx.draw_networkx_nodes(
    G, pos,
    nodelist=[n for n, d in G.nodes(data=True) if d.get("node_type") == "program"],
    node_size=60,
    node_color="lightgrey",
    alpha=0.8
)

# Axis nodes
for a in AXES:
    nx.draw_networkx_nodes(
        G, pos,
        nodelist=[a],
        node_size=1200,
        node_color=AXIS_COLORS[a],
        label=a
    )

# Edges by axis
for a in AXES:
    edges = [(u, v) for u, v, d in G.edges(data=True) if d.get("axis") == a]
    widths = [2.5 * float(G[u][v]["weight"]) for u, v in edges]

    nx.draw_networkx_edges(
        G, pos,
        edgelist=edges,
        width=widths,
        edge_color=AXIS_COLORS[a],
        alpha=0.35
    )

# Axis labels
nx.draw_networkx_labels(
    G, pos,
    labels={a: a for a in AXES},
    font_size=12,
    font_weight="bold"
)

# ----------------------------
# Legend (fixed spacing)
# ----------------------------
leg = plt.legend(
    title="Latent Semantic Axes",
    loc="lower left",
    fontsize=11,
    labelspacing=1.25,
    borderpad=1.3,
    handletextpad=0.8
)

leg.get_title().set_fontsize(13)
leg.get_title().set_y(1.08)

# ----------------------------
# Final touches
# ----------------------------
plt.title(
    "Semantic Network of Emerging Academic Programs and Latent Axes",
    fontsize=15
)

plt.axis("off")
plt.tight_layout()

plt.savefig(
    FIG_DIR / "program_axis_bipartite_readable.png",
    dpi=DPI,
    bbox_inches="tight"
)

plt.close()

    


# ============================================================
# 4) NLP GRAPH — Axis–Axis Interaction Network (Weighted)
# ============================================================
EDGE_WEIGHT_THRESHOLD = 0.08

H = nx.Graph()
for a in AXES:
    H.add_node(a)

for a, b in combinations(AXES, 2):
    tmp = df_eval[[f"score_{a}", f"score_{b}"]].dropna()
    if not tmp.empty:
        w = float((tmp[f"score_{a}"] * tmp[f"score_{b}"]).mean())
        if w >= EDGE_WEIGHT_THRESHOLD:
            H.add_edge(a, b, weight=w)

centrality = dict(H.degree(weight="weight"))
cent_vals = np.array(list(centrality.values())) if centrality else np.array([0.0])
cent_norm = (cent_vals - cent_vals.min()) / (cent_vals.max() - cent_vals.min() + 1e-9)

node_sizes = {a: 1800 + 9000 * cent_norm[i] for i, a in enumerate(centrality.keys())}

posH = nx.circular_layout(H)

plt.figure(figsize=(9, 9))
for a in H.nodes():
    nx.draw_networkx_nodes(H, posH, nodelist=[a],
                           node_size=node_sizes.get(a, 2200),
                           node_color=AXIS_COLORS[a], alpha=0.95)

edges = list(H.edges(data=True))
nx.draw_networkx_edges(
    H, posH,
    width=[18 * float(d["weight"]) for _, _, d in edges] if edges else 1.0,
    edge_color="black", alpha=0.65
)

nx.draw_networkx_labels(H, posH, font_size=11, font_weight="bold")
if edges:
    nx.draw_networkx_edge_labels(
        H, posH,
        edge_labels={(u, v): f"{float(d['weight']):.2f}" for u, v, d in edges},
        font_size=9
    )

plt.title("Axis–Axis Semantic Interaction Network\n(mean(scoreᵢ × scoreⱼ))", fontsize=13)
plt.axis("off")
plt.tight_layout()
plt.savefig(FIG_DIR / "axis_axis_interaction_weighted.png", dpi=DPI, bbox_inches="tight")
plt.close()

# ============================================================
# 5) WORLD GEOGRAPHIC MAP — QUANTILE-BASED (INTERPRETABLE)
# ============================================================
world = gpd.read_file(WORLD_SHP)

# Normalize ISO column
if "ISO_A3" not in world.columns:
    if "ADM0_A3" in world.columns:
        world["ISO_A3"] = world["ADM0_A3"]
    else:
        raise ValueError("Shapefile missing ISO_A3/ADM0_A3 columns (cannot merge by ISO3).")

country_agg = (
    df_eval.dropna(subset=["Country", "EmergenceScore"])
    .groupby("Country", as_index=False)["EmergenceScore"]
    .mean()
    .rename(columns={"EmergenceScore": "EmergenceMean"})
)

country_agg["ISO3"] = country_agg["Country"].apply(country_to_iso3)

map_df = world.merge(country_agg, how="left", left_on="ISO_A3", right_on="ISO3")

map_df_nonnull = map_df.dropna(subset=["EmergenceMean"]).copy()
if len(map_df_nonnull) >= 4:
    map_df_nonnull["EmergenceClass"] = pd.qcut(
        map_df_nonnull["EmergenceMean"],
        q=4,
        labels=["Low emergence", "Medium emergence", "High emergence", "Very high emergence"]
    )
else:
    # fallback if not enough countries for qcut
    map_df_nonnull["EmergenceClass"] = "Insufficient data for quantiles"

map_df = map_df.merge(
    map_df_nonnull[["ISO_A3", "EmergenceClass"]],
    on="ISO_A3",
    how="left"
)

fig, ax = plt.subplots(1, 1, figsize=(16, 8))
map_df.plot(
    column="EmergenceClass",
    ax=ax,
    categorical=True,
    legend=True,
    linewidth=0.4,
    edgecolor="0.6",
    cmap="viridis",
    missing_kwds={"color": "lightgrey", "label": "No data available"}
)

ax.set_title(
    "Global Patterns of Emergence Intensity in Academic Programs\n(Quantile-based classification)",
    fontsize=14
)
ax.axis("off")
plt.tight_layout()
plt.savefig(FIG_DIR / "world_emergence_choropleth_quantiles.png", dpi=DPI, bbox_inches="tight")
plt.close()

country_agg.to_excel(FIG_DIR / "country_emergence_table_for_map.xlsx", index=False)

# ============================================================
# 6) STACKED BAR — TEMPORAL EVOLUTION OF EMERGING AXES (2020–2025)
# ============================================================
df_time = df_eval.copy()

# Extract numeric year
df_time["Year"] = (
    df_time[YEAR_COL]
    .astype(str)
    .str.extract(r"(20\d{2})")[0]
    .astype(float)
)

df_time = df_time.dropna(subset=["Year"])
df_time["Year"] = df_time["Year"].astype(int)

df_time = df_time[(df_time["Year"] >= 2020) & (df_time["Year"] <= 2025)]

score_cols = [f"score_{a}" for a in AXES]
score_mat = df_time[score_cols].to_numpy(dtype=float)

# Robust dominant-axis: rows with all-NaN => DominantAxis = NaN
row_all_nan = np.isnan(score_mat).all(axis=1)
dominant_idx = np.nanargmax(np.where(np.isnan(score_mat), -np.inf, score_mat), axis=1)
dominant_axis = np.array(AXES, dtype=object)[dominant_idx]
dominant_axis[row_all_nan] = np.nan

df_time["DominantAxis"] = dominant_axis
df_time = df_time.dropna(subset=["DominantAxis"])

stack_data = (
    df_time
    .groupby(["Year", "DominantAxis"])
    .size()
    .unstack(fill_value=0)
)

for a in AXES:
    if a not in stack_data.columns:
        stack_data[a] = 0

stack_data = stack_data[AXES].sort_index()

fig, ax = plt.subplots(figsize=(9, 5))
bottom = np.zeros(len(stack_data), dtype=float)

for axis in AXES:
    values = stack_data[axis].to_numpy(dtype=float)
    ax.bar(
        stack_data.index,
        values,
        bottom=bottom,
        label=axis,
        color=AXIS_COLORS[axis],
        edgecolor="white",
        linewidth=0.6
    )
    bottom += values

ax.set_xlabel("Launch Year", fontsize=11)
ax.set_ylabel("Number of Academic Programs", fontsize=11)
ax.set_title(
    "Temporal Evolution of Emerging Academic Disciplines\n"
    "Based on Dominant Latent Semantic Axes (2020–2025)",
    fontsize=13,
    pad=12
)
ax.legend(
    title="Dominant Latent Axis",
    fontsize=9,
    title_fontsize=10,
    frameon=False
)
ax.set_xticks(list(stack_data.index))
ax.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / "stacked_bar_temporal_axes.png", dpi=DPI, bbox_inches="tight")
plt.close()

# ============================================================
# ============================================================
# 7) CONCEPTUAL FRAMEWORK — WIDE BASE & SHORTER HEIGHT PYRAMID
# ============================================================

import textwrap
from matplotlib.patches import Polygon, Patch
import matplotlib.pyplot as plt

# 🔹 Bigger figure so pyramid fully fits
fig, ax = plt.subplots(figsize=(14, 13))

# ----------------------------
# PYRAMID LAYERS (TOP → BOTTOM)
# ORANGE → LIGHT RED GRADIENT
# (TOP TWO LAYERS SHORTENED)
# ----------------------------
layers = [
    ("Global Drivers\n& Missions\n(GenAI,\nClimate, Health)", "#b30000"),
    ("Latent Disciplinary Axes\n(AI, Sustainability, Cyber)", "#d7301f"),
    ("Emergence Intensity\n(Game-theoretic Emergence Score)", "#ef6548"),
    ("Capabilities & Skills\n(Technical, Analytical, Research, Industry)", "#fc8d59"),
    ("Program Design & Interdisciplinarity\n(Observed Program Structures)", "#fdbb84"),
    ("Global & National Context\n(World Map, Regional Patterns)", "#fdd49e"),
]

n = len(layers)

# ----------------------------
# GEOMETRY CONTROLS
# ----------------------------
base_width = 6.5
layer_height = 0.8

# ----------------------------
# DRAW TRUE PYRAMID (BOTTOM → TOP)
# ----------------------------
for i, (label, color) in enumerate(layers[::-1]):
    level = i

    w_bottom = base_width * (1 - level / n)
    w_top = base_width * (1 - (level + 1) / n)

    y0 = level * layer_height
    y1 = (level + 1) * layer_height

    poly = Polygon(
        [
            (-w_bottom / 2, y0),
            ( w_bottom / 2, y0),
            ( w_top / 2, y1),
            (-w_top / 2, y1),
        ],
        closed=True,
        facecolor=color,
        edgecolor="black",
        linewidth=1.1
    )
    ax.add_patch(poly)

    wrapped = "\n".join(textwrap.wrap(label, width=44))
    ax.text(
        0,
        (y0 + y1) / 2,
        wrapped,
        ha="center",
        va="center",
        fontsize=12,
        fontweight="bold",
        linespacing=1.25
    )

# ----------------------------
# SIDE ANNOTATIONS
# ----------------------------
ax.text(
    -3.8, 2.8,
    "Comparative Analysis\n(Countries & Regions)",
    fontsize=12,
    ha="left",
    va="center"
)

ax.text(
    3.1, 1.9,
    "NLP-based Semantic Analysis\n(Embeddings & Axes)",
    fontsize=12,
    ha="left",
    va="center"
)

# ----------------------------
# LEGEND — OUTSIDE, BOTTOM RIGHT
# ----------------------------
legend_handles = [
    Patch(facecolor=color, edgecolor="black", label=label.split("\n")[0])
    for label, color in layers
]

ax.legend(
    handles=legend_handles,
    title="Conceptual Layers",
    loc="lower left",
    bbox_to_anchor=(1.02, 0.02),
    frameon=False,
    fontsize=11,
    title_fontsize=12
)

# ----------------------------
# FINAL FORMATTING
# ----------------------------
ax.set_xlim(-3.5, 3.5)
ax.set_ylim(0, n * layer_height + 0.4)
ax.axis("off")

ax.set_title(
    "Conceptual Framework of Emerging Academic Disciplines\n"
    "Integrating NLP, Game Theory, and Comparative Higher Education Analysis",
    fontsize=14,
    fontweight="bold",
    pad=22,
    loc="center"
)

plt.tight_layout()
plt.savefig(
    FIG_DIR / "conceptual_framework_pyramid_wide_short.png",
    dpi=DPI,
    bbox_inches="tight"
)
plt.close()



print("ALL ANALYSIS & VISUALIZATIONS COMPLETED SUCCESSFULLY.")
print("Figures saved in:", FIG_DIR)


Done.
Input:  D:\Edare\Article\data.xlsx
Saved:  D:\Edare\Article\cleaned_dataset.parquet
Saved:  D:\Edare\Article\cleaned_dataset.xlsx
Saved:  D:\Edare\Article\cleaned_dataset.csv
Report: D:\Edare\Article\preprocessing_report.json



Batches:   0%|          | 0/8 [00:00<?, ?it/s]

DONE (methodology only).
ALL ANALYSIS & VISUALIZATIONS COMPLETED SUCCESSFULLY.
Figures saved in: D:\Edare\Article\figures_q1
